# ML/AI — Classifying Periodic States from Sampled Histograms

We use the simulator to generate histograms from periodic states with different periods,
and train a lightweight classifier to guess the period class from sampled data.

If `scikit-learn` is unavailable, we fall back to a simple NumPy logistic regression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from quantum_hybrid_system import PeriodicState

# ---- Data generation ----
n = 9
N = 2**n
classes = [3, 4, 5, 6]  # period classes
shots_per_example = 256
examples_per_class = 60

def make_hist(samples, bins=N//4):
    hist = np.zeros(bins, dtype=float)
    for s in samples:
        hist[(s * bins) // N] += 1
    hist /= hist.sum() if hist.sum() else 1.0
    return hist

X, y = [], []
for r in classes:
    st = PeriodicState(num_qubits=n, period=r)
    for _ in range(examples_per_class):
        samples = st.measure(num_shots=shots_per_example, use_qft=True)
        X.append(make_hist(samples, bins=64))
        y.append(classes.index(r))
X = np.vstack(X)
y = np.array(y)

# ---- Train/test split ----
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
split = int(0.8 * len(X))
tr, te = idx[:split], idx[split:]
Xtr, Xte = X[tr], X[te]
ytr, yte = y[tr], y[te]

# ---- Try scikit-learn, else fallback ----
acc = None
try:
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=200)
    clf.fit(Xtr, ytr)
    acc = (clf.predict(Xte) == yte).mean()
    print("sklearn logistic regression accuracy:", acc)
except Exception as e:
    print("sklearn not available -> numpy fallback:", e)
    # Simple gradient descent logistic regression (multinomial via one-vs-rest)
    K = len(classes)
    W = np.zeros((K, Xtr.shape[1]))
    b = np.zeros(K)
    lr = 0.5
    for epoch in range(300):
        logits = W @ Xtr.T + b[:,None]
        # softmax
        ex = np.exp(logits - logits.max(axis=0, keepdims=True))
        P = ex / ex.sum(axis=0, keepdims=True)
        # one-hot
        Y = np.eye(K)[ytr].T
        # gradients
        dW = (P - Y) @ Xtr / Xtr.shape[0]
        db = (P - Y).mean(axis=1)
        W -= lr * dW
        b -= lr * db
    logits_te = W @ Xte.T + b[:,None]
    pred = logits_te.argmax(axis=0)
    acc = (pred == yte).mean()
    print("numpy softmax regression accuracy:", acc)

# ---- Quick visualization ----
plt.figure()
plt.plot(np.mean(Xtr[ytr==0], axis=0), label=f"r={classes[0]}")
plt.plot(np.mean(Xtr[ytr==1], axis=0), label=f"r={classes[1]}")
plt.plot(np.mean(Xtr[ytr==2], axis=0), label=f"r={classes[2]}")
plt.plot(np.mean(Xtr[ytr==3], axis=0), label=f"r={classes[3]}")
plt.legend()
plt.title("Mean histogram per class (train)")
plt.xlabel("coarse bin index")
plt.ylabel("probability")
plt.show()